In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Load Data
df = pd.read_csv('/content/winequality.csv')

print("--- Data Overview ---")
print(df.head())
print("\nShape:", df.shape)
print("\nMissing Values Before Imputation:")
print(df.isnull().sum())

# 2. Separate Features (X) and Target (y)
X = df.drop('quality', axis=1)
y = df['quality']

# 3. Train-Test Split (Do this FIRST to prevent data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

# 4. Impute Missing Values (Fit on train, transform on train/test)
imputer = SimpleImputer(strategy='median')
cols_to_impute = ['chlorides', 'pH', 'sulphates']

# Use .copy() to avoid SettingWithCopyWarning in pandas
X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

X_train_imputed[cols_to_impute] = imputer.fit_transform(X_train_imputed[cols_to_impute])
X_test_imputed[cols_to_impute] = imputer.transform(X_test_imputed[cols_to_impute])

# 5. Scale the Data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Convert back to DataFrame to keep column names for Feature Importance later
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

# 6. Build and Train the Model
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,             # Loosened depth to allow better learning
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',  # Handles the imbalanced quality scores
    random_state=42
)

model.fit(X_train_scaled, y_train)

# 7. Evaluate the Model (Bug removed)
train_pred = model.predict(X_train_scaled)
test_pred = model.predict(X_test_scaled)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

print("\n--- Model Performance ---")
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Testing Accuracy: {test_acc:.4f}")
print(f"Gap (Train - Test): {(train_acc - test_acc):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

# 8. Feature Importance
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\n--- Feature Importance ---")
print(importance)

# 9. Test on Existing Samples
print("\n--- Sample Predictions ---")
predictions = model.predict(X_test_scaled.iloc[:10])
for i in range(10):
    print(f"Actual: {y_test.iloc[i]} | Predicted: {predictions[i]}")

# 10. Predict on a Brand New Wine
new_wine = pd.DataFrame({
    'fixed acidity': [7.4],
    'volatile acidity': [0.70],
    'citric acid': [0.00],
    'residual sugar': [1.9],
    'chlorides': [0.076],
    'free sulfur dioxide': [11.0],
    'total sulfur dioxide': [34.0],
    'density': [0.9978],
    'pH': [3.51],
    'sulphates': [0.56],
    'alcohol': [9.4]
})

# Impute and scale the new data using the ALREADY FITTED imputer and scaler
new_wine[cols_to_impute] = imputer.transform(new_wine[cols_to_impute])
new_wine_scaled = scaler.transform(new_wine)

new_prediction = model.predict(new_wine_scaled)
print("\n--- New Wine Prediction ---")
print("Predicted Wine Quality:", new_prediction[0])

--- Data Overview ---
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0           8.94             0.507         0.29            0.90      0.031   
1           8.42             0.681         0.19            3.33      0.012   
2           9.65             0.362         0.35            2.24      0.159   
3           5.63             0.278         0.50            0.90      0.099   
4           6.49             0.474         0.59            6.77      0.142   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                  7.8                  21.9   0.9993  3.44       0.62   
1                  1.0                  53.6   0.9953  3.36       1.10   
2                 21.4                  62.2   0.9968  3.46       0.68   
3                  6.3                  19.1   0.9952  3.50       0.72   
4                 19.3                  44.9   0.9972  3.41       0.67   

   alcohol  quality  
0     9.89        5  
1    11.77        6 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: